# SAR Oil Spill — Preprocessing Notebook

Run this **once**. It only computes normalization statistics (mean/std) and saves the train/val/test split.
No patches are generated — dataset.py crops on-the-fly during training.

**Output:** `train_stats.json` + `splits.json` saved to Drive (~5KB total)

**No GPU needed** — keep runtime as CPU to save your GPU quota.

---
## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
def gb(x): return x / (1024**3)
total, used, free = shutil.disk_usage('/content')
print(f'Colab disk — total: {gb(total):.1f} GB | free: {gb(free):.1f} GB')

---
## Cell 2 — Locate Dataset

The dataset is shared at:
https://drive.google.com/drive/folders/1vKhn6wthK5ITHp3kCPBsGFvPufRPwrNW

In [ ]:
import os

DATASET_PATH = '/content/drive/MyDrive/Geo_Spill_Data'

---
## Cell 3 — Clone GitHub Repository

In [ ]:
import os

GITHUB_URL = 'https://github.com/TigranBoyakhchyan/GeoSpill-AI'
REPO_NAME  = 'GeoSpill-AI'

if os.path.exists(f'/content/{REPO_NAME}'):
    %cd /content/{REPO_NAME}
    !git pull origin main
else:
    !git clone {GITHUB_URL}
    %cd /content/{REPO_NAME}

print(f'Working directory: {os.getcwd()}')

---
## Cell 4 — Install Dependencies

In [ ]:
!pip install -q rasterio
import rasterio
print(f'rasterio: {rasterio.__version__}')

---
## Cell 5 — Compute Stats

Reads each `.tif` file once from Drive to compute mean/std.
Generates a tiny `train_stats.json` and `splits.json` — no patches, no disk space issues.

Expected time: **5-15 minutes** for 1200 images.

In [ ]:
import sys
sys.path.insert(0, '/content/oil_spill_detection')

import src.preprocess as pre

# Point directly at Drive — files are read once, so Drive speed is fine
pre.IMAGES_DIR  = f'{DATASET_PATH}/images'
pre.MASKS_DIR   = f'{DATASET_PATH}/masks'
pre.STATS_FILE  = 'data/train_stats.json'
pre.SPLITS_FILE = 'data/splits.json'

os.makedirs('data', exist_ok=True)
pre.run_preprocessing()

---
## Cell 6 — Save Stats to Drive

Saves `train_stats.json` and `splits.json` to Drive (~5KB total).
The training notebook will load these directly from Drive.

**Run before closing the session.**

In [ ]:
import shutil, os, json

SAVE_DIR = '/content/drive/MyDrive/Geo_Spill_results'
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy('data/train_stats.json', f'{SAVE_DIR}/train_stats.json')
shutil.copy('data/splits.json',      f'{SAVE_DIR}/splits.json')

# Print a summary
with open('data/train_stats.json') as f:
    stats = json.load(f)
with open('data/splits.json') as f:
    splits = json.load(f)

print(f'Saved to: {SAVE_DIR}')
print(f'  train_stats.json  — mean: {stats["mean"]}, std: {stats["std"]}')
print(f'  splits.json       — {len(splits["train"])} train | {len(splits["val"])} val | {len(splits["test"])} test images')
print('\nDone. You can now run the training notebook.')